# 03 — Probability-vector integration

If you have probability outputs from one or more *external* classifiers (e.g. popV's per-method probability matrices), you can use scBoost as a meta-learner directly on those probabilities. popV is used here as a concrete example, but the same pattern works for any annotator that emits per-class probabilities.

In [ ]:
import numpy as np
import pandas as pd, anndata as ad
import scipy.sparse as sp
from scboost_ensemble import build_probs_h5ad, StackingEnsembleDonorMetaCached

## Save a synthetic 'popV-style' probability .npz

Each method produces an (n_cells, n_classes) probability matrix.

In [ ]:
rng = np.random.default_rng(0)
n, C = 200, 5
cell_ids = np.array([f'c{i:04d}' for i in range(n)])
# 3 external classifiers each producing softmax over 5 classes
logits1, logits2, logits3 = rng.normal(size=(3, n, C))
from scipy.special import softmax
probs1 = softmax(logits1, axis=1).astype(np.float32)
probs2 = softmax(logits2, axis=1).astype(np.float32)
probs3 = softmax(logits3, axis=1).astype(np.float32)
np.savez_compressed('/tmp/scb_probs.npz', cell_ids=cell_ids,
                     popV_prob_methodA=probs1,
                     popV_prob_methodB=probs2,
                     popV_prob_methodC=probs3)
# Minimal reference h5ad
ref = ad.AnnData(X=sp.csr_matrix((n, 1), dtype=np.float32),
                  obs=pd.DataFrame({'cell_type': np.random.choice(list('ABCDE'), n),
                                     'batch':    np.random.choice([f'd{i}' for i in range(5)], n)},
                                    index=cell_ids))
ref.write_h5ad('/tmp/scb_probref.h5ad')
print('written')

## Build minimal h5ad with prob matrices as obsm entries

In [ ]:
adata = build_probs_h5ad(
    npz_path='/tmp/scb_probs.npz',
    reference_h5ad='/tmp/scb_probref.h5ad',
    out='/tmp/scb_probs_input.h5ad',
    prefix='popV_prob_',
)
list(adata.obsm.keys())

## Train scBoost meta-learners directly on the probabilities

In [ ]:
stacker = StackingEnsembleDonorMetaCached(
    obsm_keys=list(adata.obsm.keys()),
    cv_scheme='donor', n_folds=4,
    early_stopping_rounds=5,
    device='cpu', use_batch_feature=False,
)
adata.obs['split'] = 'train'
stacker.fit(adata)
print('done. cache stats:', stacker.cache_stats_)

**Note:** When the external classifiers ARE the base learners (e.g. popV), you may want to skip the scBoost base-learner stage and train ONE XGBoost directly on the concatenated probabilities — see project `260519_popV_embeddings_vs_scBoost` for that variant.